![Your 6-week roadmap](../../assets/roadmap/week2.png)

# Week 2 Exercise - SOLUTION - Tune the Chunking and Beat the Baseline nDCG

This is the **reference solution**. Try the exercise yourself first: the experiments are where the learning is!

On Day 5 you measured GradeWise's retrieval: with BM25 over the pro chunks, the baseline is **nDCG@5 = 0.861** (MRR 0.840). You also saw where it struggles: `spanning` and `holistic` questions, whose answers don't fit in one chunk.

Emma's question for you: *"Is 1,000 characters actually the right chunk size for my notes?"* Your job is to find out, with numbers.

### 🎯 Your mission
1. 🔬 Run chunking experiments (`chunk_size`, `chunk_overlap`, `contextual_header`) and **beat nDCG@5 = 0.861**
2. 📊 Find out *which categories* improve and which get worse
3. ⚖️ Check the result is fair: bigger chunks mean the LLM reads more text, so compare at a similar context budget
4. 🧪 Bonus: write your own chunker and score it

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Everything here runs <b>offline and free</b>: each experiment builds a throw-away Chroma store in a temp folder with <code>DeterministicFakeEmbedding</code> and scores it with <b>BM25</b>, which ignores the embeddings, so <b>the scores are real</b>. They measure keyword retrieval only; Part 5 shows how to confirm your winner with real embeddings and hybrid search. Don't edit any files in <code>gradewise/</code> until Part 5.</span>
        </td>
    </tr>
</table>

💡 **Fast & cheap run.** Two lines in `.env` decide what this notebook costs. `MODEL_MODE=free` runs on OpenRouter's free models for $0 (same `OPENROUTER_API_KEY`, no credits, but only about 50 requests a day), and `DATA_SIZE=small` uses a small slice of the data (2 PDFs · Q1–Q3 · 2 students) when you just want to see the APIs work. Restart the kernel after editing `.env`; the next cell prints what you're running.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

from gradewise import datasets
from gradewise.config import settings

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"ℹ️ {key_name} not set. That's OK for most of this lab: the offline cells use fakes.")
        print("   The few cells that call a real model will tell you so. Setup: Part 5 of the setup guide.")

print(settings.describe_models())
print(datasets.describe())

## Part 1: The experiment harness (provided)

Run this cell. It gives you two functions:
- `evaluate_chunking(chunk_size, chunk_overlap, contextual_header, k)` → chunk the four PDFs with those settings and score them on all 60 questions
- `evaluate_chunks(chunks, k)` → score *any* list of chunks you make yourself (for the bonus)

Each experiment takes about a second.

💡 Everything here is offline and free, so keep `DATA_SIZE=full` for this exercise: with `small` the harness scores only 2 PDFs and 5 questions, enough to check your code runs but too few to tell two chunkings apart. The numbers quoted in the solution are for `full`.

In [ ]:
import importlib.util
import sys
import tempfile
import uuid

import matplotlib.pyplot as plt
from langchain_core.documents import Document
from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise.config import DATA_DIR, REPO_ROOT
from gradewise.ingestion import chunk_documents, get_vectorstore, load_pdf
from gradewise.rag import get_retriever

spec = importlib.util.spec_from_file_location("week2_eval", REPO_ROOT / "week2" / "evaluation" / "eval.py")
evaluation = importlib.util.module_from_spec(spec)
sys.modules["week2_eval"] = evaluation
spec.loader.exec_module(evaluation)

TESTS = datasets.eval_slice(evaluation.load_tests())  # 💡 all 60 questions (the first 5 with DATA_SIZE=small)
PAGES = [page for pdf in datasets.kb_files(DATA_DIR / "knowledge-base") for page in load_pdf(pdf)]  # 💡 loaded once
TMP = tempfile.mkdtemp(prefix="gradewise_w2_exercise_")


def evaluate_chunks(chunks: list[Document], k: int = 5) -> dict:
    """Score any list of chunks with BM25 on every test question in TESTS.

    ⚠️ Offline: the Chroma store gets FAKE embeddings, but BM25 never uses them, so the scores are real.
    Returns eval.py's summary ({category: {"mrr", "ndcg", "coverage", "source", "n"}}) plus
    "context_chars": the average number of characters retrieved per question (what the LLM would read).
    """
    name = f"exp_{uuid.uuid4().hex[:8]}"
    store = get_vectorstore(name, embeddings=DeterministicFakeEmbedding(size=32), persist_dir=TMP)
    for i, chunk in enumerate(chunks):
        chunk.metadata.setdefault("chunk_index", i)
    store.add_documents(chunks, ids=[f"c{i}" for i in range(len(chunks))])
    retriever = get_retriever(store=store, k=k, mode="bm25")
    results, chars = [], 0
    for test in TESTS:
        docs = retriever.invoke(test.question)
        results.append(evaluation.evaluate_retrieval(test, docs, k))
        chars += sum(len(d.page_content) for d in docs)
    summary = evaluation.summarize(TESTS, results)
    summary["context_chars"] = chars / len(TESTS)
    return summary


def evaluate_chunking(
    chunk_size: int = 1000, chunk_overlap: int = 150, contextual_header: bool = True, k: int = 5
) -> dict:
    """One experiment: chunk the knowledge base with these settings, then score it."""
    chunks = chunk_documents(
        PAGES, chunk_size=chunk_size, chunk_overlap=chunk_overlap, contextual_header=contextual_header
    )
    summary = evaluate_chunks(chunks, k=k)
    summary["n_chunks"] = len(chunks)
    return summary


def row(name: str, summary: dict) -> dict:
    """Flatten a summary into one table row."""
    out = {"experiment": name, "chunks": summary.get("n_chunks"), "context_chars": round(summary["context_chars"])}
    out["MRR"] = round(summary["ALL"]["mrr"], 3)
    out["nDCG"] = round(summary["ALL"]["ndcg"], 3)
    for category in ("direct_fact", "numerical", "comparison", "spanning", "holistic"):
        out[f"nDCG·{category}"] = round(summary[category]["ndcg"], 2)
    return out


def show(rows: list[dict], sort_by: str | None = "nDCG") -> None:
    """Print rows as an aligned table, best first. (💡 a tiny stand-in for a pandas DataFrame)"""
    rows = sorted(rows, key=lambda r: r[sort_by], reverse=True) if sort_by else rows
    columns = list(rows[0])
    widths = {c: max(len(c), *(len(str(r[c])) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    print("  ".join("-" * widths[c] for c in columns))
    for r in rows:
        print("  ".join(str(r[c]).ljust(widths[c]) for c in columns))


print(f"✅ Harness ready: {len(TESTS)} questions, {len(PAGES)} pages, scratch stores in {TMP}")

## Part 2: Reproduce the baseline

First, make sure the harness agrees with `eval.py --offline --pipeline bm25` from Day 5. You should see nDCG = 0.861.

In [ ]:
baseline = evaluate_chunking()  # the defaults from gradewise/ingestion/chunking.py: 1000 / 150 / header on
results = [row("baseline 1000/150", baseline)]
show(results)

## Part 3: The experiments

A grid over chunk size and overlap, plus the header switched off at the baseline size.

In [ ]:
experiments = [(size, overlap, True) for size in (300, 500, 700, 1000, 1500, 2000, 3000) for overlap in (0, 150)]
experiments += [(1500, 300, True), (1000, 150, False), (1500, 150, False)]

summaries = {"baseline 1000/150": baseline}
for size, overlap, header_on in experiments:
    name = f"{size}/{overlap}" + ("" if header_on else " no header")
    if name in summaries or (size, overlap, header_on) == (1000, 150, True):
        continue
    summaries[name] = evaluate_chunking(size, overlap, header_on)
    results.append(row(name, summaries[name]))

show(results)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for overlap, colour in ((0, "#22d3ee"), (150, "#7c5cff")):
    names = [(r["experiment"].removeprefix("baseline "), r["nDCG"]) for r in results]
    points = sorted(
        (int(name.split("/")[0]), ndcg)
        for name, ndcg in names
        if "no header" not in name and name.split("/")[1] == str(overlap)
    )
    ax.plot([x for x, _ in points], [y for _, y in points], marker="o", color=colour, label=f"overlap {overlap}")
ax.axhline(0.861, color="#ef4444", ls="--", label="baseline 0.861")
ax.set_xlabel("chunk_size (characters)")
ax.set_ylabel("nDCG@5 (BM25, 60 questions)")
ax.set_title("Bigger chunks help keyword nDCG, up to a point")
ax.legend()
plt.show()

**What the grid shows** (your exact numbers match these, because BM25 and the data are deterministic):
- Small chunks (300 to 500) **lose badly** (nDCG around 0.72 to 0.81): a definition and its explanation get split, and many keywords land in a neighbouring chunk outside the top 5.
- Around **1,500 characters** the curve peaks (0.866 with the header, 0.880 without), a couple of points above the baseline. Past that, bigger adds nothing (2,000 to 3,000 sit around 0.86): most of Emma's sections are under 3,000 characters, so the chunks stop growing.
- **Overlap** matters at the baseline size (1000/150 beats 1000/0 by about 2 points): without it, a sentence cut at a boundary is lost from both chunks.
- The **contextual header** no longer helps BM25 here: *without* it is slightly better at both 1,000 and 1,500 characters. Since the tokenizer now stems words ("reliable" matches "reliability"), chunks match their topic without help, and repeating the unit title in every chunk only dilutes BM25's term statistics. The header can still earn its place for dense embeddings, which is exactly why you'd re-check with real embeddings before switching it off. Measure, don't assume.

## Part 4: Which categories moved?

In [ ]:
best_name = max(results, key=lambda r: r["nDCG"])["experiment"]
best = summaries[best_name]


def categories(summary: dict) -> dict:
    return {c: v for c, v in summary.items() if isinstance(v, dict)}


print(f"Best experiment: {best_name}")
evaluation.print_summary({"baseline": categories(baseline), best_name: categories(best)}, k=5)

**Analysis.** The gain comes almost entirely from the `holistic` questions (about 0.46 → 0.64): bigger chunks hold more of a big-picture answer, so more of its keywords make the top 5. `spanning` actually dips a little and `direct_fact` doesn't move: a bigger chunk is a blurrier match for one precise fact, and BM25's length normalisation penalises long chunks. `comparison` and `numerical` were already near the ceiling. A single `ALL` number hides all of that, which is why `eval.py` reports per category.

## Part 5: Is it a fair win?

In [ ]:
budget = {
    "1000 chars, k=5 (baseline)": baseline,
    "500 chars, k=10": evaluate_chunking(500, 100, k=10),
    f"{best_name}, k=5 (best)": best,
    "1000 chars, k=7": evaluate_chunking(1000, 150, k=7),
}
show(
    [
        {
            "set-up": name,
            "context_chars": round(s["context_chars"]),
            "nDCG": round(s["ALL"]["ndcg"], 3),
            "Keywords": round(s["ALL"]["coverage"], 3),
        }
        for name, s in budget.items()
    ],
    sort_by=None,
)

**Verdict.** The best set-up does send the LLM about 17% more text than the baseline. But it's a fair win: giving the baseline an even *bigger* budget (k = 7, more characters than the winner) only reaches about 0.865, and many small chunks with k = 10 are clearly worse (about 0.82), because the pieces of each answer are scattered over more ranks. Two honest conclusions:

1. Moving from 1,000 to about 1,500 characters (and, for BM25, dropping the header) is a real, if small, improvement for Emma's notes, mostly on big-picture questions.
2. ⚠️ Keyword nDCG is biased towards big chunks. Before changing the default, confirm with real embeddings and hybrid search (dense embeddings of very large chunks get *blurrier*), and with the LLM judge (`eval.py --answers`), which measures what actually matters: the answer.

## Part 6: Bonus, a one-chunk-per-section chunker

In [ ]:
def section_chunker(pages: list[Document]) -> list[Document]:
    """Glue each ## section back together across pages: one chunk per section, however long."""
    chunks: list[Document] = []
    for doc in chunk_documents(pages, chunk_size=100_000, chunk_overlap=0):
        key = (doc.metadata["source"], doc.metadata["section"])
        if chunks and (chunks[-1].metadata["source"], chunks[-1].metadata["section"]) == key:
            body = doc.page_content.split("\n", 1)[-1]  # 💡 drop the repeated contextual header
            chunks[-1].page_content += "\n" + body
        else:
            chunks.append(Document(doc.page_content, metadata=dict(doc.metadata)))
    return chunks


sections = section_chunker(PAGES)
mine = evaluate_chunks(sections)
mine3 = evaluate_chunks(sections, k=3)
print(f"{len(sections)} section chunks (longest {max(len(c.page_content) for c in sections):,} chars)")
print(f"k=5: nDCG = {mine['ALL']['ndcg']:.3f}, context = {mine['context_chars']:,.0f} chars/question")
print(f"k=3: nDCG = {mine3['ALL']['ndcg']:.3f}, context = {mine3['context_chars']:,.0f} chars/question")

🎉 A strong contender: **one chunk per section, with k = 3**, scores about 0.89, as well as the best fixed size, while sending the LLM *less* text (about 4,100 characters per question against 4,600). Emma's sections are short and focused, so "the whole section" is a natural unit of evidence. Before you crown it, the caveats:
- Keyword nDCG loves big chunks, and these are the biggest (up to ~3,000 characters). A **dense** vector for a whole section is a blurrier summary than one for a paragraph; check with real embeddings.
- Citations get coarser: "§1.6 Deadlocks" instead of the exact paragraph, which matters for grading evidence.
- It only works because the sections are short. A textbook with 20-page sections would blow the prompt budget; you'd fall back to size-capped chunks, which is precisely what `chunk_documents` does.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">The package keeps its default of 1,000 / 150 with contextual headers for now: it's about two points behind the best fixed size, keeps prompts small, gives the grader paragraph-level citations, and still works on long documents. But these numbers make a strong case for testing 1,500 and section-level chunks with real embeddings. If you have a key, run the "confirm it with real embeddings" steps with your favourite setting: that's the number that should decide.</span>
        </td>
    </tr>
</table>